In [ ]:
import re
import string
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
from bs4 import BeautifulSoup

import nltk
nltk.download("punkt")
nltk.download("punkt_tab")

from nltk.tokenize import word_tokenize

print("Libraries loaded successfully.")

In [ ]:
from pathlib import Path

print('BBC News Train.csv')

for file in Path(".").iterdir():
    print(file)

In [ ]:
import pandas as pd

DATA_PATH = Path("BBC News Train.csv")

df_raw = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df_raw.shape)

In [ ]:
df_raw.head()

In [ ]:
print(df_raw.columns.tolist())

In [ ]:
print("Missing values:")
print(df_raw.isnull().sum())

In [ ]:
print("Duplicate rows:", df_raw.duplicated().sum())

In [ ]:
print("Duplicate ArticleIds:", df_raw['ArticleId'].duplicated().sum())

In [ ]:
print("Categories:")
print(df_raw['Category'].value_counts())

In [ ]:
import matplotlib.pyplot as plt

df_raw['Category'].value_counts().plot(kind='bar')

plt.title('BBC News Article Distribution by Category')
plt.xlabel('Category')
plt.ylabel('Number of Articles')
plt.xticks(rotation=45)
plt.show()

In [ ]:
df_raw['text_length'] = df_raw['Text'].str.len()
print(df_raw['text_length'].describe())

In [ ]:
for i in range(3):
    print(f"\n--- Article {i} ---")
    print("Category:", df_raw.loc[i, 'Category'])
    print("Text:", df_raw.loc[i, 'Text'][:1000])

In [ ]:
df = df_raw[['ArticleId', 'Text', 'Category']].copy()
print("Working dataset shape:", df.shape)
df.head()

In [ ]:
import re

def clean_text(text):
    text = str(text)
    text = BeautifulSoup(text, "html.parser").get_text(separator=" ")
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text
print("Text cleaning function created successfully.")

In [ ]:
df = df_raw.copy()
df["Clean_Text"] = df["Text"].apply(clean_text)
print("Text cleaning completed.")

In [ ]:
display(df[["Text", "Clean_Text"]].head())

In [ ]:
df["Tokens"] = df["Clean_Text"].apply(word_tokenize)
print("Tokenization completed.")

In [ ]:
CHUNK_SIZE = 100

def chunk_text(tokens, chunk_size=CHUNK_SIZE):
    chunks = []
    for i in range(0, len(tokens), chunk_size):
        chunk = tokens[i:i + chunk_size]
        if chunk:
            chunks.append(" ".join(chunk))
    return chunks
df["Passages"] = df["Tokens"].apply(chunk_text)
print("Article chunking completed.")

In [ ]:
passage_records = []
for _, row in df.iterrows():
    for i, passage in enumerate(row["Passages"]):
        passage_records.append({
            "ArticleId": row["ArticleId"],
            "Category": row["Category"],
            "PassageId": f"{row['ArticleId']}_{i}",
            "Passage": passage
        })
df_passages = pd.DataFrame(passage_records)
print("Passage dataset created successfully.")
print("Shape:", df_passages.shape)
display(df_passages.head())

In [ ]:
df_passages.to_csv(
    "BBC_News_cleaned_chunked.csv",
    index=False
)
print("Final cleaned and chunked dataset saved successfully.")

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
display(df_passages)

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
display(df_raw[['ArticleId', 'Category', 'Text']])

In [ ]:
for index, row in df.head(5).iterrows():
    print(f"\n--- Article {index} ---")
    print(f"Category: {row['Category']}")
    print(f"Text: {row['Clean_Text']}")